In [1]:
from load_data import load_raw_data
from clean_data import clean_data
from split_data import split_data
from evaluate_model import evaluate_model
from config import RANDOM_STATE
from build_pipeline import objective,build_pipeline
from feature_engineering import build_new_features
import optuna
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

In [3]:
# Loading raw data
dataset=load_raw_data()

2026-06-12 23:57:50,535 | INFO | load_data | Raw dataset loaded.


In [4]:
# Cleaning data
dataset_cleaned=clean_data(dataset)

2026-06-12 23:57:50,540 | INFO | clean_data | Dropped customerID.
2026-06-12 23:57:50,542 | INFO | clean_data | Converted TotalCharges to numeric.
2026-06-12 23:57:50,547 | INFO | clean_data | Rows containing NaN before cleaning: 11
2026-06-12 23:57:50,554 | INFO | clean_data | Removed 11 rows with invalid TotalCharges. Remaining rows with NaN: 0.


In [5]:
# Add new features
#dataset_cleaned=build_new_features(dataset_cleaned)

In [6]:
dataset_cleaned

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7028,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7029,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7030,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [7]:
# Splitting data
X_train, X_test, y_train, y_test=split_data(dataset_cleaned, random_state=RANDOM_STATE)

2026-06-12 23:57:50,589 | INFO | split_data | Data split into train/test set with 0.8/0.2 proportion and random_state=0.


### XGBoost

In [6]:
# Creating optuna study for XGBoost
study_XGB = optuna.create_study(direction="maximize")

[I 2026-06-05 23:24:00,032] A new study created in memory with name: no-name-637008a9-f145-42c7-9d7f-2c9ef527279b


In [7]:
# Optimizing optuna study for XGBoost
study_XGB.optimize(lambda trial: objective(trial, X_train, y_train, model='XGBoost'),n_trials=100)

[I 2026-06-05 23:24:02,326] Trial 0 finished with value: 0.6089014570583148 and parameters: {'xgb_learning_rate': 0.04950826597699876, 'xgb_max_depth': 6, 'xgb_min_child_weight': 11.305173682351104, 'xgb_gamma': 2.4059121458821275, 'xgb_subsample': 0.5425456080482953, 'xgb_colsample_bytree': 0.588149471076959, 'xgb_colsample_bylevel': 0.8728096410294818, 'xgb_reg_alpha': 0.7327250708786655, 'xgb_reg_lambda': 0.0015925128506076385, 'xgb_scale_pos_weight': 3.862929150330417}. Best is trial 0 with value: 0.6089014570583148.
[I 2026-06-05 23:24:03,957] Trial 1 finished with value: 0.5725645795920804 and parameters: {'xgb_learning_rate': 0.05172964089466819, 'xgb_max_depth': 3, 'xgb_min_child_weight': 9.924092997043044, 'xgb_gamma': 0.037096558394648094, 'xgb_subsample': 0.6734162664352636, 'xgb_colsample_bytree': 0.758489873264707, 'xgb_colsample_bylevel': 0.9519547634156208, 'xgb_reg_alpha': 9.067613944257433, 'xgb_reg_lambda': 29.918688351875335, 'xgb_scale_pos_weight': 12.54933333756779

In [8]:
print("Best score for XGB:", study_XGB.best_value)
print("Best params for XGB:", study_XGB.best_params)

Best score for XGB: 0.6367339364164897
Best params for XGB: {'xgb_learning_rate': 0.02120413389867368, 'xgb_max_depth': 6, 'xgb_min_child_weight': 15.748006226450924, 'xgb_gamma': 9.229417517658616, 'xgb_subsample': 0.738962612004723, 'xgb_colsample_bytree': 0.7307626107901888, 'xgb_colsample_bylevel': 0.961552143778768, 'xgb_reg_alpha': 1.1951285767684778, 'xgb_reg_lambda': 5.1357995473243803e-08, 'xgb_scale_pos_weight': 1.984588264220078}


In [9]:
# Fitting Logistic Regression pipeline with the best trial
pipe_XGB=build_pipeline(trial=study_XGB.best_trial, model= 'XGBoost')
fitted_pipe_XGB=pipe_XGB.fit(X_train, y_train)

### LightGBM

In [6]:
# Creating optuna study for LightGBM
study_LGBM = optuna.create_study(direction="maximize")

[I 2026-06-05 00:03:31,856] A new study created in memory with name: no-name-6868c481-b484-4626-ad23-9fd1d3fe3dce


In [7]:
# Optimizing optuna study for LightGBM
study_LGBM.optimize(lambda trial: objective(trial, X_train, y_train, model='LightGBM'),n_trials=100)

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.189896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.243208 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.149981 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

...

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-06-05 00:08:20,458] Trial 2 finished with value: 0.5973863389848756 and parameters: {'lgbm_learning_rate': 0.025135320877542732, 'lgbm_max_depth': 10, 'lgbm_num_leaves': 503, 'lgbm_min_child_samples': 71, 'lgbm_subsample': 0.9678657543702787, 'lgbm_colsample_bytree': 0.9249253494381747, 'lgbm_reg_alpha': 5.266486392238566, 'lgbm_reg_lambda

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[W 2026-06-05 00:08:28,702] Trial 3 failed with parameters: {'lgbm_learning_rate': 0.015865336698271375, 'lgbm_max_depth': 7, 'lgbm_num_leaves': 84, 'lgbm_min_child_samples': 41, 'lgbm_subsample': 0.8305370358277393, 'lgbm_colsample_bytree': 0.744038854750514, 'lgbm_reg_alpha': 0.04898642704240225, 'lgbm_reg_lambda': 0.18064302986657754} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1682, in _get_outputs
    yield from self._retrieve()
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1800, in _retrieve
    time.sleep(0.01)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func

In [ ]:
print("Best score for LGBM:", study_LGBM.best_value)
print("Best params for LGBM:", study_LGBM.best_params)

In [ ]:
# Fitting LightGBM pipeline with the best trial
pipe_LGBM=build_pipeline(trial=study_LR.best_trial, model= 'LightGBM')
fitted_pipe_LGBM=pipe_LGBM.fit(X_train, y_train)

### Logistic Regression

In [8]:
# Creating optuna study for Logistic Regression
study_LR = optuna.create_study(direction="maximize")

[I 2026-06-12 23:57:50,592] A new study created in memory with name: no-name-452bc8d4-662f-4167-b546-90e1f05676ed


In [9]:
# Optimizing optuna study for Logistic Regression
study_LR.optimize(lambda trial: objective(trial, X_train, y_train, model='Logistic Regression'),n_trials=100)

[I 2026-06-12 23:57:51,946] Trial 0 finished with value: 0.628379358718019 and parameters: {'solver': 'liblinear', 'C': 0.08401154810503696}. Best is trial 0 with value: 0.628379358718019.
[I 2026-06-12 23:57:53,117] Trial 1 finished with value: 0.6317751726211478 and parameters: {'solver': 'newton-cholesky', 'C': 0.9690801125283424}. Best is trial 1 with value: 0.6317751726211478.
[I 2026-06-12 23:57:54,536] Trial 2 finished with value: 0.632291841475064 and parameters: {'solver': 'saga', 'C': 30.490899327122772}. Best is trial 2 with value: 0.632291841475064.
[I 2026-06-12 23:57:55,723] Trial 3 finished with value: 0.6304214976480692 and parameters: {'solver': 'lbfgs', 'C': 1.4822853541777288}. Best is trial 2 with value: 0.632291841475064.
[I 2026-06-12 23:57:56,918] Trial 4 finished with value: 0.6297444118990647 and parameters: {'solver': 'newton-cholesky', 'C': 0.11598271094846363}. Best is trial 2 with value: 0.632291841475064.
[I 2026-06-12 23:57:57,007] Trial 5 finished with v

In [10]:
print("Best score for LR:", study_LR.best_value)
print("Best params for LR:", study_LR.best_params)

Best score for LR: 0.6324543015853048
Best params for LR: {'solver': 'newton-cholesky', 'C': 38.41716504543486}


In [11]:
# Fitting Logistic Regression pipeline with the best trial
pipe_LR=build_pipeline(trial=study_LR.best_trial, model= 'Logistic Regression')
fitted_pipe_LR=pipe_LR.fit(X_train, y_train)

### K-Nearest Neighbors

In [12]:
# Creating optuna study for K-Nearest Neighbors
study_KNN = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:14,203] A new study created in memory with name: no-name-c86b3ddf-161a-4c8a-b89f-0bf3f6612586


In [13]:
# Optimizing optuna study for K-Nearest Neighbors
study_KNN.optimize(lambda trial: objective(trial, X_train, y_train, model='K-Nearest Neighbors'),n_trials=100)

[I 2026-06-12 23:58:14,449] Trial 0 finished with value: 0.5720073367425262 and parameters: {'knn_n_neighbors': 18, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,566] Trial 1 finished with value: 0.46806975908339493 and parameters: {'knn_n_neighbors': 4, 'knn_weights': 'uniform', 'knn_metric': 'minkowski', 'knn_p': 2}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,672] Trial 2 finished with value: 0.517102079400695 and parameters: {'knn_n_neighbors': 3, 'knn_weights': 'distance', 'knn_metric': 'euclidean', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,790] Trial 3 finished with value: 0.5440783447200765 and parameters: {'knn_n_neighbors': 12, 'knn_weights': 'uniform', 'knn_metric': 'euclidean', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:15,028] Trial 4 finished with value: 0.6002708044687809 and p

In [14]:
print("Best score for KNN:", study_KNN.best_value)
print("Best params for KNN:", study_KNN.best_params)

Best score for KNN: 0.6031276199829867
Best params for KNN: {'knn_n_neighbors': 47, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 2}


In [15]:
# Fitting K-Nearest Neighbors pipeline with the best trial
pipe_KNN=build_pipeline(trial=study_KNN.best_trial, model= 'K-Nearest Neighbors')
fitted_pipe_KNN=pipe_KNN.fit(X_train, y_train)

### Support Vector Machine

In [18]:
# Creating optuna study for Support Vector Machine
study_SVM = optuna.create_study(direction="maximize")

[I 2026-06-05 19:47:54,790] A new study created in memory with name: no-name-180d694f-a739-4ec2-a529-07f1bd4cc500


In [19]:
# Optimizing optuna study for Support Vector Machine
study_SVM.optimize(lambda trial: objective(trial, X_train, y_train, model='Support Vector Machine'),n_trials=100)

[I 2026-06-05 19:47:57,085] Trial 0 finished with value: 0.596526081066813 and parameters: {'svc_C': 0.045226555476095345, 'svc_kernel': 'linear', 'svc_gamma': 6.4627416431187e-05, 'svc_degree': 4}. Best is trial 0 with value: 0.596526081066813.
[I 2026-06-05 19:48:02,080] Trial 1 finished with value: 0.6074161706691956 and parameters: {'svc_C': 0.002449641166797178, 'svc_kernel': 'rbf', 'svc_gamma': 0.04536150494034136, 'svc_degree': 2}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:16,768] Trial 2 finished with value: 0.5233796065965034 and parameters: {'svc_C': 0.007682607615114961, 'svc_kernel': 'poly', 'svc_gamma': 1.28925952540971, 'svc_degree': 4}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:19,296] Trial 3 finished with value: 0.621037937087001 and parameters: {'svc_C': 0.0012656583498744595, 'svc_kernel': 'poly', 'svc_gamma': 0.3364510615205489, 'svc_degree': 2}. Best is trial 3 with value: 0.621037937087001.
[I 2026-06-05 19:48:24

In [ ]:
print("Best score for SVM:", study_SVM.best_value)
print("Best params for SVM:", study_SVM.best_params)

In [ ]:
# Fitting Support Vector Machine pipeline with the best trial
pipe_SVM=build_pipeline(trial=study_SVM.best_trial, model= 'Support Vector Machine')
fitted_pipe_SVM=pipe_SVM.fit(X_train, y_train)

### Decision Tree

In [16]:
# Creating optuna study for Decision Tree
study_DT = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:37,414] A new study created in memory with name: no-name-ac1c3189-fc0e-4080-abee-1d8a03c6188d


In [17]:
# Optimizing optuna study for Decision Tree
study_DT.optimize(lambda trial: objective(trial, X_train, y_train, model='Decision Tree'),n_trials=100)

[I 2026-06-12 23:58:37,489] Trial 0 finished with value: 0.580203628528124 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 38, 'dt_min_samples_split': 20, 'dt_min_samples_leaf': 3, 'dt_max_features': 'sqrt'}. Best is trial 0 with value: 0.580203628528124.
[I 2026-06-12 23:58:37,567] Trial 1 finished with value: 0.5975765251262065 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 33, 'dt_min_samples_split': 19, 'dt_min_samples_leaf': 7, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5975765251262065.
[I 2026-06-12 23:58:37,634] Trial 2 finished with value: 0.5800631849833937 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 38, 'dt_min_samples_split': 6, 'dt_min_samples_leaf': 5, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5975765251262065.
[I 2026-06-12 23:58:37,704] Trial 3 finished with value: 0.5735688017430028 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 16, 'dt_min_samples_split': 13, 'dt_min_samples_leaf': 6, 'dt_m

In [18]:
print("Best score for DT:", study_DT.best_value)
print("Best params for DT:", study_DT.best_params)

Best score for DT: 0.6142241404458695
Best params for DT: {'dt_criterion': 'gini', 'dt_max_depth': 3, 'dt_min_samples_split': 20, 'dt_min_samples_leaf': 9, 'dt_max_features': None}


In [19]:
# Fitting Decision Tree pipeline with the best trial
pipe_DT=build_pipeline(trial=study_DT.best_trial, model= 'Decision Tree')
fitted_pipe_DT=pipe_DT.fit(X_train, y_train)

### Random Forest

In [20]:
# Creating optuna study for Random Forest
study_RF = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:44,964] A new study created in memory with name: no-name-6a62cebe-51d5-4bf3-a506-67877b3d6064


In [21]:
# Optimizing optuna study for Random Forest
study_RF.optimize(lambda trial: objective(trial, X_train, y_train, model='Random Forest'),n_trials=100)

[I 2026-06-12 23:58:51,044] Trial 0 finished with value: 0.5798547695110595 and parameters: {'rf_n_estimators': 1000, 'rf_criterion': 'gini', 'rf_max_depth': 10, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 10, 'rf_max_features': None, 'rf_bootstrap': False}. Best is trial 0 with value: 0.5798547695110595.
[I 2026-06-12 23:58:52,878] Trial 1 finished with value: 0.635609807203499 and parameters: {'rf_n_estimators': 900, 'rf_criterion': 'gini', 'rf_max_depth': 46, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 10, 'rf_max_features': 'sqrt', 'rf_bootstrap': False}. Best is trial 1 with value: 0.635609807203499.
[I 2026-06-12 23:58:58,725] Trial 2 finished with value: 0.5267264432495887 and parameters: {'rf_n_estimators': 600, 'rf_criterion': 'entropy', 'rf_max_depth': 39, 'rf_min_samples_split': 12, 'rf_min_samples_leaf': 1, 'rf_max_features': None, 'rf_bootstrap': False}. Best is trial 1 with value: 0.635609807203499.
[I 2026-06-12 23:58:59,639] Trial 3 finished with value: 0.6

In [22]:
print("Best score for RF:", study_RF.best_value)
print("Best params for RF:", study_RF.best_params)

Best score for RF: 0.6398899546223423
Best params for RF: {'rf_n_estimators': 700, 'rf_criterion': 'gini', 'rf_max_depth': 43, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 9, 'rf_max_features': 'log2', 'rf_bootstrap': True}


In [23]:
# Fitting Random Forest pipeline with the best trial
pipe_RF=build_pipeline(trial=study_RF.best_trial, model= 'Random Forest')
fitted_pipe_RF=pipe_RF.fit(X_train, y_train)

### Model performances

In [23]:
# Evaluate model performance
metrics_XGB=evaluate_model(fitted_pipe_XGB,X_test,y_test)
print(metrics_XGB['report'])

NameError: name 'fitted_pipe_XGB' is not defined

In [ ]:
metrics_XGB

In [ ]:
metrics_LGBM=evaluate_model(fitted_pipe_LGBM,X_test,y_test)
print(metrics_LGBM['report'])

In [24]:
metrics_LR=evaluate_model(fitted_pipe_LR,X_test,y_test)
metrics_LR

2026-06-13 00:01:33,956 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for LogisticRegression classification model.


{'classifier_name': 'LogisticRegression',
 'accuracy': 0.7533759772565742,
 'roc_auc': 0.8516987539537508,
 'f1': 0.6312433581296493,
 'precision': 0.5238095238095238,
 'recall': 0.7941176470588235,
 'average_precision': 0.6579641398980581,
 'confusion_matrix': array([[763, 270],
        [ 77, 297]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.91      0.74      0.81      1033\n       Churn       0.52      0.79      0.63       374\n\n    accuracy                           0.75      1407\n   macro avg       0.72      0.77      0.72      1407\nweighted avg       0.81      0.75      0.77      1407\n'}

In [25]:
metrics_KNN=evaluate_model(fitted_pipe_KNN,X_test,y_test)
metrics_KNN

2026-06-13 00:01:34,116 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for KNeighborsClassifier classification model.


{'classifier_name': 'KNeighborsClassifier',
 'accuracy': 0.8052594171997157,
 'roc_auc': 0.8474123962706617,
 'f1': 0.6215469613259669,
 'precision': 0.6428571428571429,
 'recall': 0.6016042780748663,
 'average_precision': 0.6440945448671801,
 'confusion_matrix': array([[908, 125],
        [149, 225]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.86      0.88      0.87      1033\n       Churn       0.64      0.60      0.62       374\n\n    accuracy                           0.81      1407\n   macro avg       0.75      0.74      0.75      1407\nweighted avg       0.80      0.81      0.80      1407\n'}

In [ ]:
metrics_SVM=evaluate_model(fitted_pipe_SVM,X_test,y_test)
print(metrics_SVM['report'])

In [26]:
metrics_DT=evaluate_model(fitted_pipe_DT,X_test,y_test)
metrics_DT

2026-06-13 00:01:34,157 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for DecisionTreeClassifier classification model.


{'classifier_name': 'DecisionTreeClassifier',
 'accuracy': 0.7455579246624022,
 'roc_auc': 0.825116088853917,
 'f1': 0.6183368869936035,
 'precision': 0.5141843971631206,
 'recall': 0.7754010695187166,
 'average_precision': 0.5724740321549325,
 'confusion_matrix': array([[759, 274],
        [ 84, 290]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.90      0.73      0.81      1033\n       Churn       0.51      0.78      0.62       374\n\n    accuracy                           0.75      1407\n   macro avg       0.71      0.76      0.71      1407\nweighted avg       0.80      0.75      0.76      1407\n'}

In [27]:
metrics_RF=evaluate_model(fitted_pipe_RF,X_test,y_test)
metrics_RF

2026-06-13 00:01:34,455 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for RandomForestClassifier classification model.


{'classifier_name': 'RandomForestClassifier',
 'accuracy': 0.7647476901208244,
 'roc_auc': 0.8511344870606872,
 'f1': 0.6301675977653631,
 'precision': 0.5412667946257198,
 'recall': 0.7540106951871658,
 'average_precision': 0.6636940545214198,
 'confusion_matrix': array([[794, 239],
        [ 92, 282]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.90      0.77      0.83      1033\n       Churn       0.54      0.75      0.63       374\n\n    accuracy                           0.76      1407\n   macro avg       0.72      0.76      0.73      1407\nweighted avg       0.80      0.76      0.78      1407\n'}